# Notebook 4: ElasticNet — Best of Both Worlds

**Difficulty:** ⭐⭐⭐⭐ | **Estimated Time:** 2.5 hours  
**Theme:** California-style House Price Prediction  
**Week 8 — Regularization, Hyperparameter Tuning & Optimization**

---

## What You Will Learn
1. Why neither Lasso nor Ridge alone is always sufficient
2. How ElasticNet combines L1 + L2 penalties
3. The coordinate descent algorithm behind ElasticNet
4. How `l1_ratio` controls the blend from Ridge to Lasso
5. How to tune both hyperparameters (α, l1_ratio) with a 2D grid search
6. When to use ElasticNet vs Ridge vs Lasso (decision guide)

---

## Real-World Analogy First

> **The Smart Packer**
>
> You're packing for a month-long trip. You have two extreme strategies:
>
> - **Lasso strategy:** Be ruthless — only pack the absolute essentials. Leave anything that seems redundant at home (zero it out). Problem: you packed three similar t-shirts but left out all socks because "you already have sandals" (arbitrary choice among correlated items).
>
> - **Ridge strategy:** Pack everything, but each item gets shrunk — fold every item smaller. Problem: your bag is still somewhat heavy, and you brought 8 pairs of nearly-identical grey socks.
>
> - **ElasticNet strategy:** Leave behind truly useless items (L1 sparsity: no formal suit for a beach trip), AND keep the surviving items reasonably compact (L2 shrinkage: fold each item efficiently). You get a lean, well-organised bag.
>
> In house price prediction: ElasticNet **zeros out** truly irrelevant features (like `agent_id`) while distributing weight fairly across correlated feature groups (`sqft`, `sqft_log`, `sqft_per_room`).

---

## Why ElasticNet Exists: Fixing Lasso and Ridge's Weaknesses

| Weakness | Lasso (L1) | Ridge (L2) | ElasticNet Fixes It |
|----------|-----------|-----------|---------------------|
| Correlated features | Arbitrarily picks ONE, zeros out others | Keeps all with similar weights | Groups correlated features together |
| Feature selection | Yes (sparse) | No (all survive) | Yes — keeps relevant, drops irrelevant |
| More features than samples (p > n) | Selects at most n features | Works fine | Selects up to n features, more stably |
| Hyperparameter tuning | 1 parameter (α) | 1 parameter (α) | 2 parameters (α, l1_ratio) — harder |
| Correlated feature groups | Ignores the group structure | Implicitly handles | Explicitly handles with L2 grouping effect |

**Inventors:** Hui Zou and Trevor Hastie (2005) — motivated by genomics where genes come in correlated groups.

## The Mathematics of ElasticNet

### Objective Function

$$J(\theta) = \frac{1}{2n} \sum_{i=1}^{n}(y_i - \hat{y}_i)^2 + \alpha \cdot \rho \sum_{j=1}^{p}|\theta_j| + \frac{\alpha \cdot (1-\rho)}{2} \sum_{j=1}^{p}\theta_j^2$$

**Two hyperparameters:**
- $\alpha \geq 0$: overall regularisation strength (bigger = more penalty)
- $\rho \in [0, 1]$: `l1_ratio` — the mixing parameter
  - $\rho = 0$: pure Ridge (L2 only)
  - $\rho = 1$: pure Lasso (L1 only)
  - $\rho = 0.5$: equal mix of L1 and L2

**sklearn API:**
```python
ElasticNet(alpha=0.1, l1_ratio=0.5)
```

### The Grouping Effect (L2 Bonus)

When features $x_j$ and $x_k$ are perfectly correlated ($x_j = x_k$), it can be shown that ElasticNet gives them **equal coefficients**:

$$|\hat{\theta}_j - \hat{\theta}_k| \leq \frac{1}{\alpha(1-\rho)} \|y\|_2 \cdot \sqrt{2(1 - r_{jk})}$$

As $r_{jk} \to 1$: $|\hat{\theta}_j - \hat{\theta}_k| \to 0$ — the two coefficients converge.

Lasso has no such guarantee. With correlated features, Lasso may put all weight on $x_j$ and zero out $x_k$ simply based on minor numerical differences.

### Coordinate Descent — How ElasticNet is Solved

Unlike Ridge (closed-form), ElasticNet has no analytical solution because of the L1 term. Instead, we use **coordinate descent**: update one coefficient at a time, cycling through all features repeatedly until convergence.

For each coefficient $\theta_j$:

1. Compute the partial residual: $r_j = y - X\theta + x_j \theta_j$ (prediction without feature $j$)
2. Compute the OLS update for $\theta_j$ alone: $\rho_j = x_j^\top r_j / n$
3. Apply the **soft-threshold** operator: $\theta_j = S(\rho_j / z_j, \lambda_1 / z_j)$
   - where $z_j = \|x_j\|^2/n + \lambda_2$ and $S(a, \delta) = \text{sign}(a) \cdot \max(|a| - \delta, 0)$

The soft-threshold does the L1 work: if $|\rho_j|$ is small (feature contributes little), $\theta_j$ gets set to exactly zero.

In [ ]:
# ============================================================
# CELL 1: Imports and Global Settings
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from sklearn.linear_model import ElasticNet, ElasticNetCV, Lasso, Ridge, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

# Reproducibility
np.random.seed(42)

# Seaborn theme — applied globally
sns.set_theme(style='whitegrid')

# Colour constants
C_ENET   = '#9C27B0'   # purple  — ElasticNet
C_RIDGE  = '#2196F3'   # blue    — Ridge
C_LASSO  = '#FF5722'   # orange  — Lasso
C_OLS    = '#F44336'   # red     — OLS
C_TRUTH  = '#4CAF50'   # green   — ground truth

print('Libraries loaded successfully.')
print(f'NumPy {np.__version__} | Pandas {pd.__version__}')

In [ ]:
# ============================================================
# CELL 2: Generate Synthetic House Price Dataset
# 300 samples, 30 features including 3 correlated groups
# Group 1: sqft_group  (sqft / sqft_log / sqft_per_room)
# Group 2: age_group   (age / decade / renovated)
# Group 3: location    (zip_score / school / crime)
# ============================================================
np.random.seed(42)
n_samples = 300

# ---- Correlated Group 1: Size features ----
sqft          = np.random.normal(1800, 500, n_samples).clip(500, 4500)
sqft_log      = np.log(sqft) + np.random.normal(0, 0.05, n_samples)   # r ~ 0.98 with sqft
num_rooms     = 0.003 * sqft + np.random.normal(0, 0.4, n_samples)    # r ~ 0.95 with sqft
sqft_per_room = sqft / (num_rooms + 2) + np.random.normal(0, 10, n_samples)  # derived

# ---- Correlated Group 2: Age/Time features ----
age           = np.random.uniform(1, 55, n_samples)
decade        = (age / 10).astype(int).astype(float)                  # r ~ 0.97 with age
renovated     = np.where(age > 20,                                     # older houses renovated
                          np.random.binomial(1, 0.7, n_samples),
                          np.random.binomial(1, 0.2, n_samples)).astype(float)
renovation_year = 2024 - age * 0.6 + np.random.normal(0, 4, n_samples)  # correlated with age

# ---- Correlated Group 3: Location features ----
school_rating = np.random.uniform(3, 10, n_samples)
crime_index   = 10 - school_rating + np.random.normal(0, 1.5, n_samples)  # r ~ -0.9
crime_index   = crime_index.clip(0.5, 10)
zip_score     = 0.4 * school_rating - 0.35 * crime_index + np.random.normal(0, 0.3, n_samples)

# ---- Independent features ----
garage        = np.random.randint(0, 4, n_samples).astype(float)
bathrooms     = np.random.randint(1, 5, n_samples).astype(float)
stories       = np.random.randint(1, 4, n_samples).astype(float)
pool          = np.random.binomial(1, 0.2, n_samples).astype(float)
lot_size      = np.random.normal(6000, 1500, n_samples).clip(2000, 15000)
distance_cbd  = np.random.uniform(1, 30, n_samples)
basement      = np.random.binomial(1, 0.4, n_samples).astype(float)
fireplace     = np.random.binomial(1, 0.35, n_samples).astype(float)

# ---- Irrelevant noise features ----
agent_id      = np.random.randint(1, 100, n_samples).astype(float)    # truly irrelevant
month_listed  = np.random.randint(1, 13, n_samples).astype(float)     # marginally relevant
listing_photos = np.random.randint(5, 40, n_samples).astype(float)    # weakly relevant
days_on_market = np.random.randint(1, 200, n_samples).astype(float)   # not causal (post-listing)
broker_fee    = np.random.normal(5000, 1000, n_samples)               # not a house feature
tax_year      = np.random.randint(2015, 2024, n_samples).astype(float) # irrelevant
hoa_fee       = np.random.normal(200, 100, n_samples).clip(0, 800)    # weakly relevant
walk_score    = np.random.uniform(20, 100, n_samples)                 # relevant but noisy

# ---- True Price (only the real features matter) ----
price = (
    80_000
    + 110  * sqft
    - 700  * age
    + 3    * lot_size
    + 11_000 * garage
    + 14_000 * bathrooms
    + 8_000  * stories
    + 20_000 * pool
    + 9_000  * school_rating
    - 7_000  * crime_index
    - 4_000  * distance_cbd
    + 15_000 * basement
    + 6_000  * fireplace
    + 2_000  * walk_score
    + 500    * listing_photos        # weak
    + np.random.normal(0, 25_000, n_samples)
    # agent_id, month_listed, days_on_market, broker_fee, tax_year, hoa_fee: NO effect
)

# Feature order (30 total)
feature_names = [
    # Size group (correlated)
    'sqft', 'sqft_log', 'num_rooms', 'sqft_per_room',
    # Age group (correlated)
    'age', 'decade', 'renovated', 'renovation_year',
    # Location group (correlated)
    'school_rating', 'crime_index', 'zip_score',
    # Independent relevant
    'garage', 'bathrooms', 'stories', 'pool',
    'lot_size', 'distance_cbd', 'basement', 'fireplace', 'walk_score',
    # Noise / irrelevant
    'agent_id', 'month_listed', 'listing_photos', 'days_on_market',
    'broker_fee', 'tax_year', 'hoa_fee',
    # Interaction
    'sqft_x_school', 'lot_per_room', 'bath_x_stories'
]

X_arr = np.column_stack([
    sqft, sqft_log, num_rooms, sqft_per_room,
    age, decade, renovated, renovation_year,
    school_rating, crime_index, zip_score,
    garage, bathrooms, stories, pool,
    lot_size, distance_cbd, basement, fireplace, walk_score,
    agent_id, month_listed, listing_photos, days_on_market,
    broker_fee, tax_year, hoa_fee,
    sqft * school_rating / 1000,   # interaction
    lot_size / (num_rooms + 1),    # derived
    bathrooms * stories             # interaction
])

X_df = pd.DataFrame(X_arr, columns=feature_names)
y_arr = price

print(f'Dataset: {X_df.shape[0]} samples, {X_df.shape[1]} features')
print(f'Price range: ${price.min():,.0f} – ${price.max():,.0f}')
print(f'\nCorrelated groups (should be near ±0.9):')
print(f'  sqft  <-> sqft_log:      r = {np.corrcoef(sqft, sqft_log)[0,1]:.3f}')
print(f'  sqft  <-> num_rooms:     r = {np.corrcoef(sqft, num_rooms)[0,1]:.3f}')
print(f'  age   <-> decade:        r = {np.corrcoef(age, decade)[0,1]:.3f}')
print(f'  school<-> crime_index:   r = {np.corrcoef(school_rating, crime_index)[0,1]:.3f}')

In [ ]:
# ============================================================
# CELL 3: Dataset Overview — Correlation Heatmap + Group Structure
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1) Full correlation heatmap
corr = X_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, ax=axes[0], mask=mask, cmap='RdBu_r',
            vmin=-1, vmax=1, annot=False, linewidths=0.2,
            cbar_kws={'shrink': 0.7})
axes[0].set_title('Feature Correlation Matrix\n(Red = positive, Blue = negative)', fontsize=11, fontweight='bold')
axes[0].tick_params(axis='x', rotation=90, labelsize=7)
axes[0].tick_params(axis='y', labelsize=7)

# Draw boxes around correlated groups
group_bounds = [(0, 4, 'Size\nGroup'), (4, 8, 'Age\nGroup'), (8, 11, 'Location\nGroup')]
for start, end, label in group_bounds:
    rect = plt.Rectangle((start, start), end-start, end-start,
                          fill=False, edgecolor='lime', linewidth=2.5)
    axes[0].add_patch(rect)
    axes[0].text((start + end) / 2, start - 0.8, label,
                 ha='center', va='bottom', fontsize=8, color='green', fontweight='bold')

# 2) Price distribution
axes[1].hist(price / 1e3, bins=30, color=C_ENET, edgecolor='white', alpha=0.8)
axes[1].axvline(np.median(price) / 1e3, color='red', linestyle='--',
                label=f'Median ${np.median(price)/1e3:.0f}k')
axes[1].axvline(np.mean(price) / 1e3, color='orange', linestyle=':',
                label=f'Mean ${np.mean(price)/1e3:.0f}k')
axes[1].set_xlabel('Price ($000s)', fontsize=11)
axes[1].set_ylabel('Count', fontsize=11)
axes[1].set_title('House Price Distribution', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=10)

plt.suptitle('California House Price Dataset — 30 Features, 3 Correlated Groups',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('enet_dataset_overview.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 4: Soft-Threshold Function — The Core of L1 Penalty
# S(a, delta) = sign(a) * max(|a| - delta, 0)
# ============================================================

def soft_threshold(a, delta):
    """
    Soft-threshold (proximal operator for L1 penalty).

    If |a| <= delta: returns 0  (feature gets zeroed out)
    If  a > delta:   returns a - delta  (shrunk positive)
    If  a < -delta:  returns a + delta  (shrunk negative)

    This is what makes ElasticNet / Lasso produce exact zeros.
    """
    return np.sign(a) * np.maximum(np.abs(a) - delta, 0.0)


# --- Visualise the soft-threshold function ---
a_vals  = np.linspace(-3, 3, 300)
deltas  = [0.2, 0.5, 1.0, 1.5]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: soft-threshold for different delta values
palette_st = plt.cm.plasma(np.linspace(0.1, 0.85, len(deltas)))
for delta, col in zip(deltas, palette_st):
    st_vals = soft_threshold(a_vals, delta)
    axes[0].plot(a_vals, st_vals, color=col, linewidth=2.2, label=f'δ={delta}')

# Add identity line for reference
axes[0].plot(a_vals, a_vals, 'k--', linewidth=1, alpha=0.5, label='Identity (no shrinkage)')
axes[0].axhline(0, color='gray', linewidth=0.7)
axes[0].axvline(0, color='gray', linewidth=0.7)
axes[0].set_xlabel('Input value a', fontsize=11)
axes[0].set_ylabel('S(a, δ)', fontsize=11)
axes[0].set_title('Soft-Threshold Function S(a, δ)\nValues within ±δ of zero are set to EXACTLY 0', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=10)

# Right: compare soft vs hard threshold
st_half  = soft_threshold(a_vals, 0.8)
ht_half  = np.where(np.abs(a_vals) > 0.8, a_vals, 0.0)   # hard threshold
axes[1].plot(a_vals, st_half, color=C_ENET,  linewidth=2.5, label='Soft threshold (L1/ElasticNet)')
axes[1].plot(a_vals, ht_half, color=C_RIDGE, linewidth=2.5, linestyle='--', label='Hard threshold')
axes[1].plot(a_vals, a_vals,  'k--', linewidth=1, alpha=0.4, label='No shrinkage (δ=0)')
axes[1].axhline(0, color='gray', linewidth=0.7)
axes[1].set_xlabel('Input value a', fontsize=11)
axes[1].set_ylabel('Output value', fontsize=11)
axes[1].set_title('Soft vs Hard Threshold (δ=0.8)\nSoft is continuous — hard has discontinuity', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.savefig('enet_soft_threshold.png', dpi=120, bbox_inches='tight')
plt.show()

# Quick demonstration
print('Soft-threshold examples (delta=0.5):')
for val in [0.3, 0.5, 0.8, 1.5, -1.2, -0.4]:
    result = soft_threshold(val, 0.5)
    print(f'  S({val:>5}, 0.5) = {result:>6.3f}  {"← zeroed out" if result == 0 else ""}')

In [ ]:
# ============================================================
# CELL 5: ElasticNet from Scratch — Coordinate Descent
# Update one coefficient at a time, cycling until convergence
# ============================================================

def elasticnet_coordinate_descent(
        X, y, alpha=1.0, l1_ratio=0.5, max_iter=2000, tol=1e-6):
    """
    ElasticNet via coordinate descent.

    Parameters
    ----------
    X        : (n, p) feature matrix (should be standardised)
    y        : (n,)   target vector (should be mean-centred)
    alpha    : float, overall regularisation strength
    l1_ratio : float in [0,1], 0=Ridge, 1=Lasso, 0.5=equal mix
    max_iter : maximum number of full passes over all features
    tol      : convergence tolerance

    Returns
    -------
    theta    : (p,) coefficient vector
    """
    n, p  = X.shape
    theta = np.zeros(p)      # initialise all coefficients at zero

    # Separate the combined alpha into L1 and L2 components
    lam_1 = alpha * l1_ratio          # L1 penalty coefficient
    lam_2 = alpha * (1 - l1_ratio)    # L2 penalty coefficient

    for iteration in range(max_iter):
        theta_old = theta.copy()

        for j in range(p):
            # ---- Step 1: Partial residual ----
            # What the model predicts WITHOUT feature j
            r_j = y - X @ theta + X[:, j] * theta[j]

            # ---- Step 2: Univariate OLS update for theta_j ----
            # This is the optimal theta_j ignoring all other features
            rho_j = (X[:, j] @ r_j) / n

            # ---- Step 3: L2 normalisation factor ----
            # Includes the Ridge penalty denominator
            z_j = (X[:, j] ** 2).mean() + lam_2

            # ---- Step 4: Apply soft-threshold (L1 effect) ----
            # Values near zero get zeroed out; others are shrunk
            theta[j] = soft_threshold(rho_j / z_j, lam_1 / z_j)

        # ---- Convergence check ----
        delta = np.max(np.abs(theta - theta_old))
        if delta < tol:
            break

    return theta


# --- Prepare Data ---
scaler    = StandardScaler()
X_scaled  = scaler.fit_transform(X_arr)
y_centred = y_arr - y_arr.mean()

# --- Fit with scratch implementation ---
ALPHA_TEST    = 50.0
L1_RATIO_TEST = 0.5

theta_scratch = elasticnet_coordinate_descent(
    X_scaled, y_centred, alpha=ALPHA_TEST, l1_ratio=L1_RATIO_TEST
)

# --- Verify against sklearn ---
sk_enet = ElasticNet(
    alpha=ALPHA_TEST / X_scaled.shape[0],   # sklearn uses alpha/n scaling convention
    l1_ratio=L1_RATIO_TEST,
    fit_intercept=False,
    max_iter=10000
)
sk_enet.fit(X_scaled, y_centred)

print(f'ElasticNet (alpha={ALPHA_TEST}, l1_ratio={L1_RATIO_TEST})')
print('=' * 55)
comp_df = pd.DataFrame({
    'Feature':    feature_names,
    'Scratch':    theta_scratch.round(2),
    'Sklearn':    sk_enet.coef_.round(2),
    'Zero (Scratch)': (theta_scratch == 0)
})
print(comp_df.to_string(index=False))
print(f'\nNon-zero coefficients (scratch): {(theta_scratch != 0).sum()} / {len(feature_names)}')

In [ ]:
# ============================================================
# CELL 6: l1_ratio Sweep — Coefficient Heatmap
# Rows = features, Columns = l1_ratio values
# Shows the transition from Ridge (all survive) to Lasso (sparse)
# ============================================================

l1_ratios = [0.0, 0.2, 0.5, 0.8, 1.0]
ALPHA_SWEEP = 30.0

# Collect coefficients for each l1_ratio
coef_matrix = np.zeros((len(feature_names), len(l1_ratios)))

for col_i, ratio in enumerate(l1_ratios):
    if ratio == 0.0:
        # Pure Ridge — use analytical solution
        from numpy.linalg import solve
        p = X_scaled.shape[1]
        A = X_scaled.T @ X_scaled + ALPHA_SWEEP * np.eye(p)
        coef_matrix[:, col_i] = solve(A, X_scaled.T @ y_centred)
    else:
        coef_matrix[:, col_i] = elasticnet_coordinate_descent(
            X_scaled, y_centred, alpha=ALPHA_SWEEP, l1_ratio=ratio
        )

# --- Plot heatmap ---
fig, ax = plt.subplots(figsize=(12, 9))

coef_df_hmap = pd.DataFrame(
    coef_matrix,
    index=feature_names,
    columns=[f'l1={r}\n({"Ridge" if r==0 else ("Lasso" if r==1 else "Mix")})' for r in l1_ratios]
)

sns.heatmap(
    coef_df_hmap, ax=ax, cmap='RdBu_r', center=0,
    annot=True, fmt='.0f', annot_kws={'size': 8},
    linewidths=0.5, cbar_kws={'label': 'Coefficient value', 'shrink': 0.7}
)

ax.set_title(f'ElasticNet Coefficients: l1_ratio Sweep (α={ALPHA_SWEEP})\n'
             'White/grey = near zero | Red = positive | Blue = negative',
             fontsize=12, fontweight='bold')
ax.set_xlabel('l1_ratio (0=Ridge → 1=Lasso)', fontsize=11)
ax.set_ylabel('Feature', fontsize=11)
ax.tick_params(axis='y', labelsize=8)

# Add group labels on the left
group_labels = [(1.5, 'Size\nGroup'), (5.5, 'Age\nGroup'), (9.5, 'Location\nGroup')]
for y_pos, label in group_labels:
    ax.text(-1.5, y_pos, label, ha='right', va='center',
            fontsize=8, fontweight='bold', color='darkgreen')

plt.tight_layout()
plt.savefig('enet_l1ratio_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

# Count non-zeros per l1_ratio
print('Non-zero coefficients per l1_ratio:')
for ratio, col_i in zip(l1_ratios, range(len(l1_ratios))):
    n_nonzero = (np.abs(coef_matrix[:, col_i]) > 1).sum()
    print(f'  l1_ratio={ratio}: {n_nonzero}/{len(feature_names)} non-zero')

In [ ]:
# ============================================================
# CELL 7: Correlated Feature Group Behaviour
# Compare Lasso vs ElasticNet for the size group (sqft/sqft_log/num_rooms/sqft_per_room)
# Key question: does Lasso arbitrarily zero out 3 of 4?
# ============================================================

# Fit Lasso and ElasticNet with the same total regularisation
ALPHA_COMPARE = 30.0

# Lasso (l1_ratio = 1.0)
theta_lasso = elasticnet_coordinate_descent(
    X_scaled, y_centred, alpha=ALPHA_COMPARE, l1_ratio=1.0
)

# Ridge (l1_ratio = 0.0)
p = X_scaled.shape[1]
theta_ridge = np.linalg.solve(
    X_scaled.T @ X_scaled + ALPHA_COMPARE * np.eye(p),
    X_scaled.T @ y_centred
)

# ElasticNet (l1_ratio = 0.5)
theta_enet = elasticnet_coordinate_descent(
    X_scaled, y_centred, alpha=ALPHA_COMPARE, l1_ratio=0.5
)

# Focus on the 3 correlated groups
group_features = {
    'Size Group': ['sqft', 'sqft_log', 'num_rooms', 'sqft_per_room'],
    'Age Group':  ['age', 'decade', 'renovated', 'renovation_year'],
    'Location Group': ['school_rating', 'crime_index', 'zip_score']
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax_i, (group_name, group_feats) in enumerate(group_features.items()):
    ax = axes[ax_i]
    idxs = [feature_names.index(f) for f in group_feats]

    x_pos = np.arange(len(group_feats))
    width = 0.25

    ax.bar(x_pos - width, theta_lasso[idxs], width=width,
           color=C_LASSO, alpha=0.85, label='Lasso')
    ax.bar(x_pos,         theta_ridge[idxs], width=width,
           color=C_RIDGE, alpha=0.85, label='Ridge')
    ax.bar(x_pos + width, theta_enet[idxs],  width=width,
           color=C_ENET,  alpha=0.85, label='ElasticNet')

    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(group_feats, rotation=30, ha='right', fontsize=9)
    ax.set_title(group_name, fontsize=11, fontweight='bold')
    ax.set_ylabel('Coefficient' if ax_i == 0 else '')
    if ax_i == 0:
        ax.legend(fontsize=9)

    # Annotate zeroed-out features
    for xi, feat_idx in enumerate(idxs):
        if abs(theta_lasso[feat_idx]) < 1:
            ax.text(xi - width, 50, 'ZERO', ha='center', fontsize=7,
                    color='darkred', fontweight='bold')

plt.suptitle(
    'Correlated Feature Groups: Lasso vs Ridge vs ElasticNet\n'
    'Lasso arbitrarily zeros out correlated features | ElasticNet distributes weight',
    fontsize=12, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig('enet_correlated_groups.png', dpi=120, bbox_inches='tight')
plt.show()

print('Lasso zeroed features (|coef| < 1):')
for feat, coef in zip(feature_names, theta_lasso):
    if abs(coef) < 1:
        print(f'  {feat}')

In [ ]:
# ============================================================
# CELL 8: Number of Non-Zero Features vs l1_ratio
# Bar chart showing sparsity as l1_ratio increases
# ============================================================

l1_ratios_sparsity = np.array([0.0, 0.1, 0.2, 0.3, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
ALPHA_SPARSITY = 25.0
ZERO_THRESHOLD = 10   # treat |coef| < 10 as zero (scaled domain)

n_nonzero = []
for ratio in l1_ratios_sparsity:
    if ratio == 0.0:
        # Ridge: analytical solution, almost nothing is exactly zero
        p = X_scaled.shape[1]
        theta_r = np.linalg.solve(
            X_scaled.T @ X_scaled + ALPHA_SPARSITY * np.eye(p),
            X_scaled.T @ y_centred
        )
        n_nonzero.append((np.abs(theta_r) >= ZERO_THRESHOLD).sum())
    else:
        theta_e = elasticnet_coordinate_descent(
            X_scaled, y_centred, alpha=ALPHA_SPARSITY, l1_ratio=ratio
        )
        n_nonzero.append((np.abs(theta_e) >= ZERO_THRESHOLD).sum())

# Bar chart
fig, ax = plt.subplots(figsize=(11, 5))

colors_bar = plt.cm.RdPu(np.linspace(0.2, 0.85, len(l1_ratios_sparsity)))
bars = ax.bar(l1_ratios_sparsity, n_nonzero, width=0.07, color=colors_bar, edgecolor='white')

# Label each bar
for bar, count in zip(bars, n_nonzero):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            str(count), ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xlabel('l1_ratio  (0 = Ridge | 1 = Lasso)', fontsize=12)
ax.set_ylabel('Number of Non-Zero Coefficients', fontsize=12)
ax.set_title(f'Sparsity vs l1_ratio  (α={ALPHA_SPARSITY}, threshold=|{ZERO_THRESHOLD}|)\n'
             'As l1_ratio increases, more features are zeroed out',
             fontsize=12, fontweight='bold')
ax.set_xticks(l1_ratios_sparsity)
ax.set_xticklabels([f'{r}\n{"(Ridge)" if r==0 else "(Lasso)" if r==1 else ""}'
                    for r in l1_ratios_sparsity], fontsize=9)
ax.set_ylim(0, len(feature_names) + 3)
ax.axhline(len(feature_names), color='gray', linestyle=':', linewidth=1, label='All 30 features')
ax.legend(fontsize=10)

# Annotate regions
ax.axvspan(0, 0.15, alpha=0.07, color=C_RIDGE, label='Ridge-dominant')
ax.axvspan(0.85, 1.0, alpha=0.07, color=C_LASSO, label='Lasso-dominant')

plt.tight_layout()
plt.savefig('enet_sparsity_bar.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 9: Regularisation Paths — All 3 Models Side by Side
# ElasticNet at l1_ratio=0.5 shows intermediate behaviour
# ============================================================

alphas_path = np.logspace(-1, 3, 80)
p = X_scaled.shape[1]

coef_ridge_path = np.zeros((len(alphas_path), p))
coef_enet_path  = np.zeros((len(alphas_path), p))
coef_lasso_path = np.zeros((len(alphas_path), p))

for i, a in enumerate(alphas_path):
    # Ridge: analytical
    coef_ridge_path[i] = np.linalg.solve(
        X_scaled.T @ X_scaled + a * np.eye(p), X_scaled.T @ y_centred
    )
    # ElasticNet l1=0.5
    coef_enet_path[i]  = elasticnet_coordinate_descent(
        X_scaled, y_centred, alpha=a, l1_ratio=0.5
    )
    # Lasso
    coef_lasso_path[i] = elasticnet_coordinate_descent(
        X_scaled, y_centred, alpha=a, l1_ratio=1.0
    )

fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=False)

palette20 = plt.cm.tab20(np.linspace(0, 1, p))

for ax, paths, title, color_main in [
        (axes[0], coef_ridge_path, f'Ridge (l1_ratio=0.0)',  C_RIDGE),
        (axes[1], coef_enet_path,  f'ElasticNet (l1_ratio=0.5)', C_ENET),
        (axes[2], coef_lasso_path, f'Lasso (l1_ratio=1.0)',  C_LASSO)]:

    for j in range(p):
        ax.plot(np.log10(alphas_path), paths[:, j],
                color=palette20[j], linewidth=1.3, alpha=0.8)

    ax.axhline(0, color='black', linewidth=0.8, linestyle=':')
    ax.set_xlabel('log₁₀(α)', fontsize=11)
    ax.set_ylabel('Coefficient' if ax == axes[0] else '')
    ax.set_title(title + '\n', fontsize=11, fontweight='bold')

    # Count non-zeros at mid-alpha
    mid_i = len(alphas_path) // 2
    nz = (np.abs(paths[mid_i]) > 5).sum()
    ax.text(0.05, 0.95, f'Non-zero at α=10:\n{(np.abs(paths[20])>5).sum()}/{p}',
            transform=ax.transAxes, fontsize=9, va='top',
            bbox=dict(boxstyle='round,pad=0.3', fc='lightyellow', alpha=0.9))

plt.suptitle('Regularisation Paths: Ridge vs ElasticNet vs Lasso\n'
             'Ridge: smooth shrinkage | ElasticNet: selective shrinkage | Lasso: aggressive zeroing',
             fontsize=12, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('enet_regularisation_paths.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 10: Two-Hyperparameter Grid Search
# alpha x l1_ratio heatmap of 5-fold CV MSE
# Find the optimal (alpha, l1_ratio) combination
# ============================================================

alphas_grid   = np.logspace(-1, 3, 15)
l1_grid       = [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9, 1.0]
cv_grid       = KFold(n_splits=5, shuffle=True, random_state=42)

mse_grid = np.zeros((len(l1_grid), len(alphas_grid)))

for i, ratio in enumerate(l1_grid):
    for j, alpha in enumerate(alphas_grid):
        if ratio == 0.0:
            # sklearn Ridge
            model = Pipeline([('sc', StandardScaler()),
                              ('m', Ridge(alpha=alpha, fit_intercept=True))])
        elif ratio == 1.0:
            # sklearn Lasso
            model = Pipeline([('sc', StandardScaler()),
                              ('m', Lasso(alpha=alpha, fit_intercept=True, max_iter=5000))])
        else:
            # sklearn ElasticNet
            model = Pipeline([('sc', StandardScaler()),
                              ('m', ElasticNet(alpha=alpha, l1_ratio=ratio,
                                               fit_intercept=True, max_iter=5000))])

        scores = cross_val_score(model, X_arr, y_arr,
                                 cv=cv_grid, scoring='neg_mean_squared_error')
        mse_grid[i, j] = -scores.mean()

# --- Heatmap ---
fig, ax = plt.subplots(figsize=(13, 6))

mse_df = pd.DataFrame(
    mse_grid / 1e9,
    index=[f'{r}\n({"Ridge" if r==0 else "Lasso" if r==1 else ""})' for r in l1_grid],
    columns=[f'{a:.1f}' for a in alphas_grid]
)

sns.heatmap(mse_df, ax=ax, cmap='YlOrRd_r', annot=True, fmt='.2f',
            annot_kws={'size': 8}, linewidths=0.3,
            cbar_kws={'label': 'CV MSE (×10⁹)', 'shrink': 0.7})

# Highlight the best cell
best_row, best_col = np.unravel_index(np.argmin(mse_grid), mse_grid.shape)
ax.add_patch(plt.Rectangle((best_col, best_row), 1, 1,
                             fill=False, edgecolor='cyan', linewidth=3, zorder=5))

ax.set_xlabel('α (Regularisation Strength)', fontsize=12)
ax.set_ylabel('l1_ratio', fontsize=12)
ax.set_title(f'2-Hyperparameter Grid: 5-Fold CV MSE (×10⁹)\n'
             f'Best: α={alphas_grid[best_col]:.2f}, l1_ratio={l1_grid[best_row]}  '
             f'(cyan box)',
             fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('enet_grid_search_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Best alpha:     {alphas_grid[best_col]:.4f}')
print(f'Best l1_ratio:  {l1_grid[best_row]}')
print(f'Best CV MSE:    {mse_grid[best_row, best_col]:.4e}')

In [ ]:
# ============================================================
# CELL 11: sklearn ElasticNetCV — Efficient 2-Parameter Tuning
# ElasticNetCV searches over alpha using coordinate descent path
# More efficient than manual GridSearchCV
# ============================================================

from sklearn.linear_model import ElasticNetCV

# Scale the data first (outside CV to understand the process;
# in production, always scale inside a Pipeline)
X_scaled_full = StandardScaler().fit_transform(X_arr)

# Search over l1_ratios and let ElasticNetCV find best alpha automatically
enet_cv = ElasticNetCV(
    l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0],
    alphas=np.logspace(-2, 3, 50),
    cv=5,
    fit_intercept=True,
    max_iter=10_000,
    random_state=42
)
enet_cv.fit(X_scaled_full, y_arr)

print('ElasticNetCV Results')
print('=' * 50)
print(f'  Best alpha:            {enet_cv.alpha_:.6f}')
print(f'  Best l1_ratio:         {enet_cv.l1_ratio_:.2f}')
print(f'  Non-zero coefficients: {(enet_cv.coef_ != 0).sum()} / {len(feature_names)}')
print(f'  Intercept:             {enet_cv.intercept_:.2f}')
print()

# Show which features survived
coef_series = pd.Series(enet_cv.coef_, index=feature_names)
surviving = coef_series[coef_series != 0].sort_values(key=abs, ascending=False)
zeroed    = coef_series[coef_series == 0].index.tolist()

print('Features that SURVIVED (non-zero):')
for feat, coef in surviving.items():
    bar = '█' * min(int(abs(coef) / 2000), 20)
    print(f'  {feat:<22}  {coef:>+10.0f}  {bar}')

print(f'\nFeatures ZEROED out ({len(zeroed)}):')
print('  ' + ', '.join(zeroed))

# Prediction quality
y_pred_cv = enet_cv.predict(X_scaled_full)
print(f'\nTraining RMSE: ${np.sqrt(mean_squared_error(y_arr, y_pred_cv)):,.2f}')

In [ ]:
# ============================================================
# CELL 12: Predicted vs Actual — All 4 Models
# OLS / Ridge / Lasso / ElasticNet side-by-side comparison
# ============================================================

# Fit all models with best alpha from grid search
BEST_ALPHA = alphas_grid[best_col]
BEST_RATIO = l1_grid[best_row]

models = [
    ('OLS',       Pipeline([('sc', StandardScaler()), ('m', LinearRegression())])),
    ('Ridge',     Pipeline([('sc', StandardScaler()), ('m', Ridge(alpha=BEST_ALPHA))])),
    ('Lasso',     Pipeline([('sc', StandardScaler()), ('m', Lasso(alpha=BEST_ALPHA, max_iter=5000))])),
    ('ElasticNet', Pipeline([('sc', StandardScaler()),
                              ('m', ElasticNet(alpha=BEST_ALPHA, l1_ratio=BEST_RATIO, max_iter=5000))]))
]

colors_model = [C_OLS, C_RIDGE, C_LASSO, C_ENET]

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes_flat = axes.flatten()

cv_mses = {}

for (name, pipe), ax, col in zip(models, axes_flat, colors_model):
    # Fit on full data
    pipe.fit(X_arr, y_arr)
    y_pred = pipe.predict(X_arr)

    # CV MSE for fair comparison
    cv_scores = cross_val_score(pipe, X_arr, y_arr,
                                cv=KFold(5, shuffle=True, random_state=42),
                                scoring='neg_mean_squared_error')
    cv_mse = -cv_scores.mean()
    cv_mses[name] = cv_mse

    train_rmse = np.sqrt(mean_squared_error(y_arr, y_pred))
    cv_rmse    = np.sqrt(cv_mse)

    ax.scatter(y_arr / 1e3, y_pred / 1e3, alpha=0.35, s=15, color=col)
    lims = [min(y_arr.min(), y_pred.min()) / 1e3,
            max(y_arr.max(), y_pred.max()) / 1e3]
    ax.plot(lims, lims, 'k--', linewidth=1.5, label='Perfect')

    ax.set_title(f'{name}\nTrain RMSE: ${train_rmse:,.0f} | CV RMSE: ${cv_rmse:,.0f}',
                 fontsize=11, fontweight='bold', color=col)
    ax.set_xlabel('Actual Price ($000s)', fontsize=10)
    ax.set_ylabel('Predicted Price ($000s)', fontsize=10)
    ax.legend(fontsize=9)

plt.suptitle('Predicted vs Actual: OLS / Ridge / Lasso / ElasticNet\n'
             f'All regularised models use α={BEST_ALPHA:.2f}',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('enet_predicted_vs_actual_all.png', dpi=120, bbox_inches='tight')
plt.show()

print('CV RMSE Summary (lower is better):')
for name, mse in sorted(cv_mses.items(), key=lambda x: x[1]):
    print(f'  {name:<12}  CV RMSE = ${np.sqrt(mse):>10,.0f}')

In [ ]:
# ============================================================
# CELL 13: Coefficient Comparison — All 4 Models
# Shows the progressive sparsification from OLS → ElasticNet → Lasso
# ============================================================

# Retrieve fitted coefficients from each pipeline
coef_store = {}
for name, pipe in models:
    pipe.fit(X_arr, y_arr)
    coef_store[name] = pipe.named_steps['m'].coef_

# Build a DataFrame for visualisation
coef_compare_df = pd.DataFrame(coef_store, index=feature_names)

fig, ax = plt.subplots(figsize=(14, 6))

x_pos = np.arange(len(feature_names))
width = 0.2

offsets = [-1.5*width, -0.5*width, 0.5*width, 1.5*width]
model_colors = [C_OLS, C_RIDGE, C_LASSO, C_ENET]
model_names  = ['OLS', 'Ridge', 'Lasso', 'ElasticNet']

for offset, color, name in zip(offsets, model_colors, model_names):
    ax.bar(x_pos + offset, coef_compare_df[name],
           width=width, color=color, alpha=0.80, label=name)

ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(feature_names, rotation=60, ha='right', fontsize=8)
ax.set_ylabel('Scaled Coefficient', fontsize=11)
ax.set_title('Coefficient Comparison: OLS vs Ridge vs Lasso vs ElasticNet\n'
             'Note: Lasso and ElasticNet produce exact zeros for irrelevant features',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

# Shade the noise feature region
noise_start = feature_names.index('agent_id')
noise_end   = feature_names.index('hoa_fee') + 1
ax.axvspan(noise_start - 0.5, noise_end - 0.5, alpha=0.07,
           color='red', label='Noise features')
ax.text((noise_start + noise_end) / 2 - 0.5, ax.get_ylim()[1] * 0.95,
        'Noise / Irrelevant\nFeatures', ha='center', va='top',
        fontsize=9, color='darkred', fontweight='bold')

plt.tight_layout()
plt.savefig('enet_coef_comparison_all.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 14: Convergence of Coordinate Descent
# Track the objective function value across iterations
# ============================================================

def elasticnet_with_history(X, y, alpha=1.0, l1_ratio=0.5, max_iter=500):
    """
    Same as elasticnet_coordinate_descent but records objective history.
    """
    n, p  = X.shape
    theta = np.zeros(p)
    lam_1 = alpha * l1_ratio
    lam_2 = alpha * (1 - l1_ratio)
    history = []

    for iteration in range(max_iter):
        for j in range(p):
            r_j     = y - X @ theta + X[:, j] * theta[j]
            rho_j   = (X[:, j] @ r_j) / n
            z_j     = (X[:, j] ** 2).mean() + lam_2
            theta[j] = soft_threshold(rho_j / z_j, lam_1 / z_j)

        # Compute ElasticNet objective value
        residuals = y - X @ theta
        mse  = 0.5 * (residuals ** 2).mean()
        l1_t = lam_1 * np.abs(theta).sum()
        l2_t = 0.5 * lam_2 * (theta ** 2).sum()
        obj  = mse + l1_t + l2_t
        history.append(obj)

    return theta, np.array(history)


# Run with history for different l1_ratios
l1_conv = [0.0, 0.3, 0.7, 1.0]
ALPHA_CONV = 15.0

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: convergence curves for different l1_ratio
conv_palette = [C_RIDGE, '#7E57C2', C_ENET, C_LASSO]
for ratio, col in zip(l1_conv, conv_palette):
    if ratio == 0.0:
        # For Ridge, compute objective at analytical solution level
        # Use a warm-start Ridge approximated via coordinate descent
        _, hist = elasticnet_with_history(X_scaled, y_centred,
                                           alpha=ALPHA_CONV, l1_ratio=0.01)
    else:
        _, hist = elasticnet_with_history(X_scaled, y_centred,
                                           alpha=ALPHA_CONV, l1_ratio=ratio)
    label = f'l1_ratio={ratio}' + (' (Ridge)' if ratio == 0 else ' (Lasso)' if ratio == 1 else '')
    axes[0].plot(np.arange(1, len(hist)+1), hist, color=col, linewidth=2.2, label=label)

axes[0].set_xlabel('Iteration (one full cycle over all features)', fontsize=11)
axes[0].set_ylabel('Objective Value', fontsize=11)
axes[0].set_title('Coordinate Descent Convergence\n(monotonically decreasing — guaranteed)',
                  fontsize=11, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].set_yscale('log')

# Right: zoom into first 50 iterations
for ratio, col in zip(l1_conv, conv_palette):
    r_use = max(ratio, 0.01)   # avoid pure ridge for visualisation
    _, hist = elasticnet_with_history(X_scaled, y_centred,
                                       alpha=ALPHA_CONV, l1_ratio=r_use, max_iter=80)
    label = f'l1_ratio={ratio}'
    axes[1].plot(np.arange(1, len(hist)+1), hist, color=col, linewidth=2.2, label=label)

axes[1].set_xlabel('Iteration', fontsize=11)
axes[1].set_ylabel('Objective Value', fontsize=11)
axes[1].set_title('First 80 Iterations — Initial Convergence Speed',
                  fontsize=11, fontweight='bold')
axes[1].legend(fontsize=9)

plt.suptitle('Coordinate Descent Convergence for ElasticNet',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('enet_convergence.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 15: Perfect Correlation Demo
# What happens when 3 features are perfectly correlated?
# Lasso: arbitrary (picks one) | ElasticNet: equal weights
# ============================================================

np.random.seed(42)
n_demo = 200

# Create a feature and two perfect copies
x_base = np.random.normal(0, 1, n_demo)
X_perfect = np.column_stack([
    x_base,                                          # original
    x_base + np.random.normal(0, 1e-10, n_demo),    # near-perfect copy 1
    x_base + np.random.normal(0, 1e-10, n_demo),    # near-perfect copy 2
])
y_perfect = 5 * x_base + np.random.normal(0, 0.3, n_demo)

# Standardise
X_perf_sc = StandardScaler().fit_transform(X_perfect)
y_perf_c  = y_perfect - y_perfect.mean()

ALPHA_PERF = 0.5
n_runs = 30   # run multiple times with tiny random perturbations to show instability

lasso_coefs_runs = np.zeros((n_runs, 3))
enet_coefs_runs  = np.zeros((n_runs, 3))

for run in range(n_runs):
    # Add tiny noise to make each run slightly different
    Xr = X_perf_sc + np.random.normal(0, 1e-6, X_perf_sc.shape)
    lasso_coefs_runs[run] = elasticnet_coordinate_descent(
        Xr, y_perf_c, alpha=ALPHA_PERF, l1_ratio=1.0
    )
    enet_coefs_runs[run] = elasticnet_coordinate_descent(
        Xr, y_perf_c, alpha=ALPHA_PERF, l1_ratio=0.5
    )

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

feat_labels = ['feature_1', 'feature_2\n(near-copy)', 'feature_3\n(near-copy)']

for ax, coefs, title, color in [
        (axes[0], lasso_coefs_runs, 'Lasso (l1_ratio=1.0)\nArbitrary, unstable selection', C_LASSO),
        (axes[1], enet_coefs_runs,  'ElasticNet (l1_ratio=0.5)\nStable grouping effect', C_ENET)]:

    # Plot all runs as faint dots
    for run_i in range(n_runs):
        ax.scatter([0, 1, 2], coefs[run_i], alpha=0.25, s=40, color=color)

    # Plot mean with error bar
    ax.errorbar([0, 1, 2], coefs.mean(axis=0), yerr=coefs.std(axis=0) * 2,
                fmt='D', color='black', capsize=6, markersize=8, linewidth=2,
                label='Mean ± 2σ', zorder=5)

    ax.axhline(0, color='gray', linewidth=0.7)
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(feat_labels, fontsize=10)
    ax.set_ylabel('Coefficient', fontsize=11)
    ax.set_title(title, fontsize=11, fontweight='bold', color=color)
    ax.legend(fontsize=9)
    ax.set_ylim(-0.5, 4)

plt.suptitle('Perfect Correlation: 3 Identical Features\n'
             f'({n_runs} runs with tiny random perturbations)',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('enet_perfect_correlation.png', dpi=120, bbox_inches='tight')
plt.show()

print('Lasso coefficient std (instability):', lasso_coefs_runs.std(axis=0).round(3))
print('ElasticNet coeff std (stability):   ', enet_coefs_runs.std(axis=0).round(3))

## Decision Guide: Which Regularisation Method to Use?

| Situation | Best Choice | Reason |
|-----------|-------------|--------|
| All features are relevant, some correlated | **Ridge** | Stabilises coefficients, keeps all features, distributes weight evenly |
| Many irrelevant features, low multicollinearity | **Lasso** | Automatic feature selection (exact zeros) |
| Many irrelevant features + correlated groups | **ElasticNet** | Selects AND handles groups fairly |
| p >> n (more features than samples) | **ElasticNet** (or Lasso) | Lasso selects at most n features; ElasticNet is more stable |
| Perfectly correlated features | **ElasticNet** or **Ridge** | Lasso is arbitrary; Ridge/ElasticNet give equal weights |
| Need exact sparsity (regulatory requirement) | **Lasso** or **ElasticNet** | Ridge never gives exact zeros |
| Genomics / gene expression data | **ElasticNet** | Genes come in correlated pathways — grouping effect is essential |
| Simple baseline, few features, no collinearity | **OLS** (Ridge if uncertain) | Regularisation adds unnecessary bias |

**Practical starting point:**
1. Try ElasticNet with `l1_ratio=[0.1, 0.5, 0.9]` and cross-validate `alpha`
2. If `l1_ratio → 0` wins: use Ridge; if `l1_ratio → 1` wins: use Lasso
3. ElasticNet at the optimal l1_ratio often beats both pure alternatives

---

## The Regularisation Family Tree

```
OLS (no regularisation)
 |
 +──── Ridge (L2):   min MSE + α·Σθⱼ²           [no zeros, stable]
 |
 +──── Lasso (L1):   min MSE + α·Σ|θⱼ|          [sparse, unstable with correlations]
 |
 +──── ElasticNet:   min MSE + α·ρ·Σ|θⱼ| + α·(1-ρ)/2·Σθⱼ²
                                                  [sparse AND stable]
                          ↑              ↑
                       L1 term        L2 term
         (feature selection)   (grouping effect)
```

In [ ]:
# ============================================================
# CELL 16: Full Pipeline — Production-Ready ElasticNet
# End-to-end: preprocessing → tuning → fitting → prediction
# ============================================================

from sklearn.model_selection import train_test_split

# 1. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_arr, y_arr, test_size=0.2, random_state=42
)

# 2. Build pipeline
enet_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', ElasticNetCV(
        l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
        alphas=np.logspace(-2, 3, 40),
        cv=5,
        fit_intercept=True,
        max_iter=10_000,
        random_state=42
    ))
])

# 3. Fit on training data only (CV is internal — no data leakage)
enet_pipeline.fit(X_train, y_train)
enet_model = enet_pipeline.named_steps['model']

# 4. Predict on held-out test set
y_pred_test = enet_pipeline.predict(X_test)
y_pred_train = enet_pipeline.predict(X_train)

train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
test_rmse  = np.sqrt(mean_squared_error(y_test,  y_pred_test))

print('Production ElasticNet Pipeline')
print('=' * 50)
print(f'  Best alpha:            {enet_model.alpha_:.6f}')
print(f'  Best l1_ratio:         {enet_model.l1_ratio_:.2f}')
print(f'  Non-zero features:     {(enet_model.coef_ != 0).sum()} / {len(feature_names)}')
print(f'  Train RMSE:            ${train_rmse:>12,.2f}')
print(f'  Test  RMSE:            ${test_rmse:>12,.2f}')
print(f'  Train/Test gap:        ${abs(test_rmse - train_rmse):>12,.2f}  (lower = less overfit)')

# 5. Feature importance plot
coef_abs = pd.Series(np.abs(enet_model.coef_), index=feature_names)
coef_abs_sorted = coef_abs[coef_abs > 0].sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
colors_feat = [C_ENET if c > 0 else '#aaa' for c in coef_abs_sorted.values]
coef_abs_sorted.plot(kind='barh', ax=ax, color=C_ENET, alpha=0.85)
ax.set_xlabel('|Coefficient| (standardised)', fontsize=11)
ax.set_title('ElasticNet Feature Importance\n(only non-zero features shown)',
             fontsize=12, fontweight='bold')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig('enet_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 17: Summary Statistics — All Models on Same Test Set
# ============================================================

from sklearn.metrics import r2_score

final_models = [
    ('OLS',       Pipeline([('sc', StandardScaler()), ('m', LinearRegression())])),
    ('Ridge',     Pipeline([('sc', StandardScaler()),
                             ('m', RidgeCV(alphas=np.logspace(-2, 3, 50), cv=5))])),
    ('Lasso',     Pipeline([('sc', StandardScaler()),
                             ('m', LassoCV(alphas=np.logspace(-2, 3, 50), cv=5, max_iter=10_000))])),
    ('ElasticNet', enet_pipeline),
]

from sklearn.linear_model import RidgeCV, LassoCV

print('Final Model Comparison (80/20 train-test split)')
print('=' * 65)
print(f'{"Model":<14} {"Train RMSE":>12} {"Test RMSE":>12} {"Test R²":>8} {"Non-zero":>10}')
print('-' * 65)

for name, pipe in final_models:
    pipe.fit(X_train, y_train)
    y_tr = pipe.predict(X_train)
    y_te = pipe.predict(X_test)

    tr_rmse = np.sqrt(mean_squared_error(y_train, y_tr))
    te_rmse = np.sqrt(mean_squared_error(y_test, y_te))
    te_r2   = r2_score(y_test, y_te)

    # Non-zero count
    coef = pipe.named_steps['m'].coef_
    if hasattr(pipe.named_steps['m'], 'best_estimator_'):
        coef = pipe.named_steps['m'].best_estimator_.coef_
    nz = (np.abs(coef) > 1).sum()

    print(f'{name:<14} ${tr_rmse:>10,.0f} ${te_rmse:>10,.0f} {te_r2:>8.3f} {nz:>10}/{len(feature_names)}')

print('-' * 65)
print('Note: Test RMSE is the honest metric. Smaller gap = less overfitting.')

## Key Takeaways

| Concept | Plain-English Summary |
|---------|----------------------|
| **ElasticNet** | Combines L1 (sparsity) + L2 (grouping) — best of both worlds |
| **l1_ratio** | 0 = Ridge, 1 = Lasso, 0.5 = equal mix |
| **Grouping effect** | Correlated features get similar (not arbitrary) coefficients |
| **Coordinate descent** | Update one θⱼ at a time using soft-threshold; iterate to convergence |
| **Soft-threshold** | Values within ±λ of zero become exactly zero (this creates sparsity) |
| **2D tuning** | Need both α and l1_ratio — use ElasticNetCV or GridSearchCV |
| **vs Lasso** | More stable with correlated features; l1_ratio=1 recovers Lasso |
| **vs Ridge** | Sparser; l1_ratio=0 recovers Ridge |

---

## Formula Reference Card

| Formula | Meaning |
|---------|--------|
| $J = \frac{1}{2n}\|y-X\theta\|^2 + \alpha\rho\|\theta\|_1 + \frac{\alpha(1-\rho)}{2}\|\theta\|_2^2$ | ElasticNet objective |
| $S(a, \delta) = \text{sign}(a)\cdot\max(|a|-\delta, 0)$ | Soft-threshold operator |
| $\theta_j \leftarrow S\!\left(\frac{\rho_j}{z_j}, \frac{\lambda_1}{z_j}\right)$ | Coordinate descent update |
| $z_j = \bar{x}_j^2 + \lambda_2$ | Normalisation factor (L2 term) |
| `ElasticNet(alpha=0.1, l1_ratio=0.5)` | sklearn API |
| `ElasticNetCV(l1_ratio=[.1,.5,.9])` | with auto alpha selection |

## Self-Check Questions

Answer these before looking at the solutions. Spend at least 2 minutes thinking through each one.

---

**Question 1:**  
You have 3 perfectly correlated features (correlation = 1.0). What will Lasso do with them? What will ElasticNet do? Which behaviour is more stable and why?

<details>
<summary><strong>Answer (click to reveal)</strong></summary>

**Lasso:** With perfectly correlated features, Lasso's objective has a flat region — infinitely many solutions achieve the same loss. The solver will effectively **pick one feature arbitrarily** (whichever it encounters first in coordinate descent, or which has a tiny numerical advantage) and set the other two to exactly zero. If you run Lasso multiple times with tiny perturbations to the data, you'll get wildly different feature selections (feature 1 on run 1, feature 3 on run 2, etc.). This is unstable and untrustworthy in practice.

**ElasticNet:** The L2 term in ElasticNet penalises the *sum of squared coefficients*, which creates a preference for solutions where the weight is **shared equally** across the correlated group. With 3 perfectly identical features where the total weight should be $w$, ElasticNet tends toward $\theta_1 = \theta_2 = \theta_3 = w/3$. This is the "grouping effect" proven by Zou and Hastie (2005).

**Which is more stable:** ElasticNet, because:
1. The optimal solution is unique (the L2 term restores strict convexity)
2. Tiny data perturbations change the coefficients slightly, not drastically
3. The selected features and their magnitudes are reproducible

The Cell 15 demo showed this directly: Lasso coefficient standard deviation across 30 runs was much higher than ElasticNet's.

</details>

---

**Question 2:**  
ElasticNet has two hyperparameters (α, l1_ratio). This makes tuning harder than Lasso or Ridge (one parameter each). What specific benefits justify this added complexity?

<details>
<summary><strong>Answer (click to reveal)</strong></summary>

The two extra benefits that justify the complexity:

**1. Stable feature selection with correlated features (Lasso cannot do this)**  
Lasso will arbitrarily pick one feature from each correlated group and zero out the rest. ElasticNet keeps the entire group with appropriately shrunk coefficients. In practice: if `sqft`, `sqft_log`, and `num_rooms` all carry the same information, Lasso might tell you only `sqft_log` matters. ElasticNet correctly tells you all three carry similar, distributed weight. This leads to more interpretable and stable models.

**2. Feature selection even when p > n (Lasso is limited here)**  
When you have more features than samples (common in genomics, NLP), Lasso can select at most $n$ features before breaking down. ElasticNet can select more features because the L2 term stabilises the problem, allowing the algorithm to keep multiple features from each correlated group.

**The complexity cost is manageable:**  
- `ElasticNetCV` searches over l1_ratio automatically
- If the optimal l1_ratio = 0 → just use Ridge
- If the optimal l1_ratio = 1 → just use Lasso  
- ElasticNet at its optimal l1_ratio often outperforms both pure alternatives

The extra hyperparameter is really a dial that **automatically discovers** whether you need Ridge-like, Lasso-like, or mixed behaviour for your specific dataset.

</details>

---

**Question 3:**  
`l1_ratio = 0` is equivalent to Ridge. `l1_ratio = 1` is equivalent to Lasso. What does `l1_ratio = 0.5` actually minimise? Write out the complete objective function.

<details>
<summary><strong>Answer (click to reveal)</strong></summary>

With `l1_ratio = 0.5` (and overall strength `α`), ElasticNet minimises:

$$J(\theta) = \frac{1}{2n} \sum_{i=1}^{n}(y_i - \hat{y}_i)^2 + \underbrace{\alpha \cdot 0.5 \cdot \sum_{j=1}^{p}|\theta_j|}_{\text{L1 penalty (Lasso term)}} + \underbrace{\frac{\alpha \cdot 0.5}{2} \cdot \sum_{j=1}^{p}\theta_j^2}_{\text{L2 penalty (Ridge term)}}$$

Which simplifies to:

$$J(\theta) = \frac{1}{2n}\|y - X\theta\|_2^2 + \frac{\alpha}{2}\|\theta\|_1 + \frac{\alpha}{4}\|\theta\|_2^2$$

In words: **the squared prediction error, plus half-alpha times the sum of absolute coefficients, plus quarter-alpha times the sum of squared coefficients.**

The L1 term (Lasso part) provides sparsity — it zeros out truly irrelevant features.  
The L2 term (Ridge part) provides the grouping effect — it keeps correlated features together.

The key insight: with `l1_ratio = 0.5`, the L1 and L2 penalties are NOT equal in magnitude (the L2 term has an extra /2). This is because the L2 term is already divided by 2 in the standard ElasticNet formulation. The equal mixing means equal weight between the L1 and L2 penalty structure, not equal numerical values.

In sklearn: `ElasticNet(alpha=α, l1_ratio=0.5)` implements exactly this formula.

</details>

## Next Steps

You have now mastered the core regularisation toolkit. Here is where to go next:

1. **Week 8 continued:** Hyperparameter tuning strategies (Grid Search, Random Search, Bayesian Optimisation) — how to efficiently search larger hyperparameter spaces.

2. **Practice:** Apply Ridge, Lasso, and ElasticNet to `sklearn.datasets.fetch_california_housing()` and compare:
   - Which features does Lasso zero out?
   - Does ElasticNet beat both?
   - What l1_ratio does ElasticNetCV select?

3. **Research:** The original ElasticNet paper (Zou & Hastie, 2005) is readable. Pay attention to the grouping effect proof and the path algorithm.

4. **Extension:** Sparse group Lasso — applies group-level L1 penalty on top of individual L1. Useful when you know the feature group structure in advance.

---

**Regularisation Quick-Reference:**

```
Problem type                    → Recommended method
─────────────────────────────────────────────────────
All features relevant            → Ridge
Many irrelevant, uncorrelated    → Lasso
Many irrelevant, correlated      → ElasticNet
p >> n                           → ElasticNet
Need exact zeros                 → Lasso or ElasticNet
Genomics / gene pathways         → ElasticNet
Text features (sparse by design) → Lasso or ElasticNet
Unknown situation                → ElasticNetCV (finds best l1_ratio)
```